# CS781 Phase 3 — Joint LoRA(q,v) + Hierarchical BiLSTM (End-to-End)

**Baseline (frozen decoder + BiLSTM):** 0.4331 dev F1 on 13 labels / 0.5118 on 11 supported

**Changes from baseline:**
- Add LoRA (r=16) on decoder q, v only — decoder adapts to the task
- End-to-end joint training (no precompute)
- Sub-batch sentences within doc + gradient checkpointing (memory)
- **Uniform sampler** with 2× num_samples (proven > weighted sampler)
- Light regularization (what worked in frozen baseline)

**Expected:** dev F1 0.46–0.52, test F1 0.50–0.56.

In [ ]:
# CELL 1 — LOAD PHASE 1 ASSETS + PHASE 3 DATA

from pathlib import Path
from google.colab import drive
from getpass import getpass
from huggingface_hub import HfApi, hf_hub_download
import os, re, sys, subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "peft"])

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")

print("Searching Drive...")
drive_files = []
for root, dirs, files in os.walk(DRIVE_ROOT):
    dirs[:] = [d for d in dirs if d not in {".Trash", ".ipynb_checkpoints", "node_modules"}]
    for filename in files:
        drive_files.append(Path(root) / filename)

def normalized_name(path):
    return re.sub(r" \(\d+\)(?=\.)", "", path.name).lower()

def locate_drive_file(filename, preferred_folder=None):
    target = filename.lower()
    matches = [p for p in drive_files if p.name.lower() == target or normalized_name(p) == target]
    if preferred_folder:
        preferred = [p for p in matches if preferred_folder.lower() in str(p.parent).lower()]
        if preferred:
            matches = preferred
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} on Drive.")
    matches.sort(key=lambda p: (p.name.lower() != target, len(p.parts), str(p).lower()))
    return matches[0]

CHECKPOINT_PATH = locate_drive_file("best.pt")
TOKENIZER_JSON  = locate_drive_file("bpe_40k.json")
TOKENIZER_PY    = locate_drive_file("a1_tokenizer.py")

HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = getpass("HF read token (hidden): ")

api = HfApi()
repo_files = api.list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
phase3_files = [f for f in repo_files if "phase_3_rr/" in f.lower()]

def download_phase3_file(filename):
    target = filename.lower()
    matches = [f for f in phase3_files if Path(f).name.lower() == target]
    if not matches:
        matches = [f for f in phase3_files if normalized_name(Path(f)).lower() == target]
    if "test" in target:
        preferred = [f for f in matches if "/test/" in f.lower()]
    elif "label_vocab" in target:
        preferred = matches
    else:
        preferred = [f for f in matches if "/train/" in f.lower()]
    repo_path = (preferred or matches)[0]
    return Path(hf_hub_download(repo_id=HF_REPO, filename=repo_path, repo_type="dataset", token=HF_TOKEN))

CL_TRAIN    = download_phase3_file("CL_train.json")
CL_DEV      = download_phase3_file("CL_dev.json")
IT_TRAIN    = download_phase3_file("IT_train.json")
IT_DEV      = download_phase3_file("IT_dev.json")
CL_TEST     = download_phase3_file("CL_test.json")
IT_TEST     = download_phase3_file("IT_test.json")
LABEL_VOCAB = download_phase3_file("label_vocab.json")
CKPT_PATH   = CHECKPOINT_PATH

if str(TOKENIZER_PY.parent) not in sys.path:
    sys.path.insert(0, str(TOKENIZER_PY.parent))

print(f"Decoder: {CKPT_PATH}")
print("All assets loaded.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Searching Drive...
HF read token (hidden): ··········
Decoder: /content/drive/MyDrive/best.pt
All assets loaded.


In [ ]:
# CELL 2 — IMPORTS & DEVICE

import json, math, random, zipfile, time
from pathlib import Path
from dataclasses import dataclass
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.amp import autocast, GradScaler

from peft import LoraConfig, get_peft_model
from a1_tokenizer import BPETokenizer

OUTPUT_DIR = DRIVE_ROOT / "CS781_Phase3_LoRA_BiLSTM_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} ({p.total_memory/1024**3:.1f} GiB)")

In [ ]:
# CELL 3 — HYPERPARAMETERS (joint config, informed by frozen baseline)

@dataclass
class TrainCfg:
    # Decoder arch (must match Phase 1)
    vocab_size: int = 40_000
    d_model: int    = 768
    n_layers: int   = 19
    n_query_heads: int = 12
    n_kv_heads: int    = 4
    d_ff: int       = 2048
    max_seq_len: int = 512
    rope_theta: float = 10_000.0

    # Sentence encoding (joint needs more memory, shorter than frozen)
    sent_max_tokens: int = 512
    sent_sub_batch: int = 8   # sentences encoded per sub-batch within a doc

    # LoRA on q, v only
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    lora_target_modules = ("q", "v")

    # Hierarchical BiLSTM (LIGHT reg — matches winning frozen config)
    num_labels: int = 13
    bilstm_hidden: int = 256
    bilstm_layers: int = 2
    bilstm_dropout: float = 0.3
    cls_dropout: float = 0.4

    # Training (joint is slower per epoch)
    epochs: int = 10
    doc_batch_size: int = 1       # one doc at a time (memory)
    grad_accum: int = 4           # effective batch = 4 docs
    train_doubling: int = 2       # uniform sampler yields 2x docs/epoch
    lora_lr: float = 2e-4
    bilstm_lr: float = 5e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.05
    max_grad_norm: float = 1.0
    class_weight_cap: float = 3.0
    label_smoothing: float = 0.0

    early_stop_patience: int = 4
    seed: int = 42

    def __post_init__(self):
        self.head_dim = self.d_model // self.n_query_heads
        self.n_groups = self.n_query_heads // self.n_kv_heads

CFG = TrainCfg()
print(f"LoRA: r={CFG.lora_r}, alpha={CFG.lora_alpha}, targets={CFG.lora_target_modules}")
print(f"Decoder: {CFG.n_layers}L GQA, sent_max={CFG.sent_max_tokens}")
print(f"Train: {CFG.epochs}ep, grad_accum={CFG.grad_accum}, patience={CFG.early_stop_patience}")
print(f"Light reg: bilstm_dropout={CFG.bilstm_dropout}, cls_dropout={CFG.cls_dropout}, wd={CFG.weight_decay}")

LoRA: r=16, alpha=32, targets=('q', 'v')
Decoder: 19L GQA, sent_max=512
Train: 10ep, grad_accum=4, patience=4
Light reg: bilstm_dropout=0.3, cls_dropout=0.4, wd=0.01


In [ ]:
# CELL 4 — SEED + TOKENIZER

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False

seed_everything(CFG.seed)

tokenizer = BPETokenizer.load(str(TOKENIZER_JSON))
BASE_VOCAB_SIZE = int(tokenizer.vocab_size)
SPECIAL_TOKEN_MAP = getattr(tokenizer, "special_tokens", {}) or {}
PAD_ID = int(SPECIAL_TOKEN_MAP.get("[PAD]", 0))
assert BASE_VOCAB_SIZE == CFG.vocab_size
print(f"Tokenizer vocab={BASE_VOCAB_SIZE}, PAD={PAD_ID}")

Tokenizer vocab=40000, PAD=0


In [ ]:
# CELL 5 — LOAD DATA + CAPPED CLASS WEIGHTS

with open(LABEL_VOCAB) as f:
    LABEL2ID = json.load(f)
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)
CFG.num_labels = NUM_LABELS

def load_train_data(path):
    with open(path) as f: data = json.load(f)
    return [(d["id"], d["text"], [LABEL2ID[l] for l in d["labels"]]) for d in data]

def load_test_data(path):
    with open(path) as f: data = json.load(f)
    return [(doc_id, sentences) for doc_id, sentences in data.items()]

cl_train_docs = load_train_data(CL_TRAIN)
it_train_docs = load_train_data(IT_TRAIN)
cl_dev_docs   = load_train_data(CL_DEV)
it_dev_docs   = load_train_data(IT_DEV)
train_docs = cl_train_docs + it_train_docs
dev_docs   = cl_dev_docs + it_dev_docs
test_docs  = load_test_data(CL_TEST) + load_test_data(IT_TEST)

print(f"Train: {len(train_docs)} docs, Dev: {len(dev_docs)}, Test: {len(test_docs)}")

all_labels = [l for _, _, labels in train_docs for l in labels]
label_counts = Counter(all_labels)
total = sum(label_counts.values())
class_weights = torch.zeros(NUM_LABELS)
for i in range(NUM_LABELS):
    freq = label_counts.get(i, 1) / total
    class_weights[i] = 1.0 / math.sqrt(freq)
class_weights = class_weights / class_weights.mean()
class_weights = torch.clamp(class_weights, max=CFG.class_weight_cap)
class_weights_tensor = class_weights.to(DEVICE)
print(f"Class weights (capped at {CFG.class_weight_cap}) ready")

Train: 80 docs, Dev: 10, Test: 10
Class weights (capped at 3.0) ready


In [ ]:
# CELL 6 — DECODER ARCHITECTURE (Phase 1 LegalLM backbone)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight

def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()

def apply_rope(x, cos, sin, offset=0):
    T = x.size(2)
    c = cos[offset:offset+T].unsqueeze(0).unsqueeze(0)
    s = sin[offset:offset+T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rotated = torch.cat([-x2, x1], dim=-1)
    c_exp = torch.cat([c.expand(-1,-1,-1,hd), c.expand(-1,-1,-1,hd)], dim=-1)
    s_exp = torch.cat([s.expand(-1,-1,-1,hd), s.expand(-1,-1,-1,hd)], dim=-1)
    return x * c_exp + rotated * s_exp

class GQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nqh=cfg.n_query_heads; self.nkh=cfg.n_kv_heads; self.ng=cfg.n_groups
        self.hd=cfg.head_dim; self.dm=cfg.d_model
        self.q=nn.Linear(cfg.d_model, cfg.n_query_heads*cfg.head_dim, bias=False)
        self.k=nn.Linear(cfg.d_model, cfg.n_kv_heads*cfg.head_dim, bias=False)
        self.v=nn.Linear(cfg.d_model, cfg.n_kv_heads*cfg.head_dim, bias=False)
        self.o=nn.Linear(cfg.n_query_heads*cfg.head_dim, cfg.d_model, bias=False)
        self.attn_drop=nn.Dropout(0.0); self.resid_drop=nn.Dropout(0.0)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)

    def forward(self, x, cos, sin, mask):
        B,T,_ = x.shape
        q = self.q(x).view(B,T,self.nqh,self.hd).transpose(1,2)
        k = self.k(x).view(B,T,self.nkh,self.hd).transpose(1,2)
        v = self.v(x).view(B,T,self.nkh,self.hd).transpose(1,2)
        q = apply_rope(q, cos, sin); k = apply_rope(k, cos, sin)
        if self.ng > 1:
            k = k.unsqueeze(2).expand(B,self.nkh,self.ng,T,self.hd).reshape(B,self.nqh,T,self.hd)
            v = v.unsqueeze(2).expand(B,self.nkh,self.ng,T,self.hd).reshape(B,self.nqh,T,self.hd)
        w = torch.matmul(q, k.transpose(-2,-1)) * self.scale
        if mask is not None:
            w = w + mask
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        out = torch.matmul(w, v).transpose(1,2).contiguous().view(B,T,self.dm)
        return self.resid_drop(self.o(out))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(0.0)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model); self.attn=GQA(cfg)
        self.norm2=RMSNorm(cfg.d_model); self.ffn=SwiGLU(cfg)
    def forward(self, x, cos, sin, mask):
        x = x + self.attn(self.norm1(x), cos, sin, mask)
        x = x + self.ffn(self.norm2(x))
        return x

class LegalDecoderBackbone(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(0.0)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len * 2, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, input_ids, padding_mask=None):
        B, T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        causal = torch.triu(torch.ones(T, T, device=input_ids.device, dtype=torch.bool), diagonal=1)
        mask = torch.zeros((T, T), device=input_ids.device, dtype=x.dtype).masked_fill(causal, float("-inf"))
        mask = mask[None, None, :, :].expand(B, 1, T, T).clone()
        if padding_mask is not None:
            mask = mask.masked_fill(padding_mask[:, None, None, :], float("-inf"))
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin, mask)
        return self.norm(x)

print("Decoder arch defined ✓")

Decoder arch defined ✓


In [ ]:
%pip install -U "torchao>0.16.0"

In [ ]:
# CELL 7 — LOAD DECODER + APPLY LORA ON q, v ONLY

decoder = LegalDecoderBackbone(CFG).to(DEVICE)

ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
state = ckpt["model"] if "model" in ckpt else ckpt
print(f"Checkpoint: step={ckpt.get('step', '?')}, val_loss={ckpt.get('val_loss', '?')}")

wanted = {k: v for k, v in state.items() if not k.startswith("lm_head")}
result = decoder.load_state_dict(wanted, strict=False)
print(f"Loaded. Missing: {len(result.missing_keys)}, Unexpected: {len(result.unexpected_keys)}")
assert len(result.missing_keys) < 5

# Freeze all; LoRA will add trainable adapters on q,v
for p in decoder.parameters():
    p.requires_grad = False

lora_config = LoraConfig(
    r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    target_modules=list(CFG.lora_target_modules),
    lora_dropout=CFG.lora_dropout,
    bias="none",
)

decoder = get_peft_model(decoder, lora_config)
decoder = decoder.to(DEVICE)

trainable = sum(p.numel() for p in decoder.parameters() if p.requires_grad)
total = sum(p.numel() for p in decoder.parameters())
print(f"\nLoRA trainable (q,v only): {trainable:,} ({trainable/total*100:.3f}%)")
decoder.print_trainable_parameters()

Checkpoint: step=21500, val_loss=2.3908054494857787
Loaded. Missing: 0, Unexpected: 0

LoRA trainable (q,v only): 778,240 (0.515%)
trainable params: 778,240 || all params: 151,065,856 || trainable%: 0.5152


In [ ]:
# CELL 8 — PRE-TOKENIZE DOCUMENTS (no gradients needed)

def tokenize_doc(sentences, max_tokens=256):
    out = []
    for s in sentences:
        ids = tokenizer.encode(s)[:max_tokens] if s else [PAD_ID]
        if not ids:
            ids = [PAD_ID]
        out.append(ids)
    return out

def preprocess_docs(docs, has_labels=True):
    out = []
    for doc in docs:
        if has_labels:
            doc_id, sentences, labels = doc
        else:
            doc_id, sentences = doc
            labels = None
        tok = tokenize_doc(sentences, CFG.sent_max_tokens)
        out.append({"doc_id": doc_id, "sent_tokens": tok, "labels": labels,
                    "n_sentences": len(sentences)})
    return out

print("Pre-tokenizing docs...")
t0 = time.time()
train_prep = preprocess_docs(train_docs, has_labels=True)
dev_prep   = preprocess_docs(dev_docs,   has_labels=True)
test_prep  = preprocess_docs(test_docs,  has_labels=False)
cl_dev_ids = {cd[0] for cd in cl_dev_docs}
it_dev_ids = {cd[0] for cd in it_dev_docs}
cl_dev_prep = [d for d in dev_prep if d["doc_id"] in cl_dev_ids]
it_dev_prep = [d for d in dev_prep if d["doc_id"] in it_dev_ids]
print(f"Done in {time.time()-t0:.1f}s")
print(f"Max sentences/doc: {max(d['n_sentences'] for d in train_prep)}")

Pre-tokenizing docs...
Done in 1.4s
Max sentences/doc: 1816


In [ ]:
# CELL 9 — DATASET + UNIFORM SAMPLER (proven > weighted sampler in baseline)

class DocDataset(Dataset):
    def __init__(self, docs): self.docs = docs
    def __len__(self): return len(self.docs)
    def __getitem__(self, i): return self.docs[i]

def doc_collate(batch):
    assert len(batch) == 1   # doc_batch_size=1
    return batch[0]

# Uniform sampler: 2x effective training per epoch, no class bias
uniform_weights = torch.ones(len(train_prep), dtype=torch.double)
sampler = WeightedRandomSampler(
    weights=uniform_weights,
    num_samples=len(train_prep) * CFG.train_doubling,
    replacement=True,
)

train_loader   = DataLoader(DocDataset(train_prep), batch_size=1, sampler=sampler, collate_fn=doc_collate)
dev_loader     = DataLoader(DocDataset(dev_prep),   batch_size=1, shuffle=False, collate_fn=doc_collate)
cl_dev_loader  = DataLoader(DocDataset(cl_dev_prep), batch_size=1, shuffle=False, collate_fn=doc_collate)
it_dev_loader  = DataLoader(DocDataset(it_dev_prep), batch_size=1, shuffle=False, collate_fn=doc_collate)
test_loader    = DataLoader(DocDataset(test_prep),  batch_size=1, shuffle=False, collate_fn=doc_collate)

print(f"Train: {len(train_loader)} docs/epoch (uniform sampler, {CFG.train_doubling}x)")
print(f"Dev: {len(dev_loader)}, Test: {len(test_loader)}")

Train: 160 docs/epoch (uniform sampler, 2x)
Dev: 10, Test: 10


In [ ]:
# CELL 10 — HIERARCHICAL BILSTM + JOINT ENCODE FUNCTION

class HierarchicalBiLSTM(nn.Module):
    def __init__(self, d_in, num_labels, hidden, layers, bilstm_dropout, cls_dropout):
        super().__init__()
        self.input_norm = nn.LayerNorm(d_in)
        self.bilstm = nn.LSTM(
            input_size=d_in, hidden_size=hidden, num_layers=layers,
            bidirectional=True, batch_first=True,
            dropout=bilstm_dropout if layers > 1 else 0.0,
        )
        self.out_dropout = nn.Dropout(cls_dropout)
        self.head = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.GELU(),
            nn.Dropout(cls_dropout),
            nn.Linear(hidden, num_labels),
        )

    def forward(self, sent_emb):
        x = self.input_norm(sent_emb)
        out, _ = self.bilstm(x)
        out = self.out_dropout(out)
        return self.head(out)


bilstm = HierarchicalBiLSTM(
    d_in=CFG.d_model,
    num_labels=CFG.num_labels,
    hidden=CFG.bilstm_hidden,
    layers=CFG.bilstm_layers,
    bilstm_dropout=CFG.bilstm_dropout,
    cls_dropout=CFG.cls_dropout,
).to(DEVICE)


def encode_doc_sentences(decoder, sent_tokens, sub_batch=8, training=True):
    """Encode all sentences of a doc. Returns (S, D) with gradients if training.

    Uses gradient checkpointing on each sub-batch to save VRAM.
    """
    all_emb = []
    for start in range(0, len(sent_tokens), sub_batch):
        chunk = sent_tokens[start:start + sub_batch]
        maxlen = max(len(c) for c in chunk)
        input_ids = torch.full((len(chunk), maxlen), PAD_ID, dtype=torch.long, device=DEVICE)
        pad_mask = torch.ones((len(chunk), maxlen), dtype=torch.bool, device=DEVICE)
        for i, ids in enumerate(chunk):
            input_ids[i, :len(ids)] = torch.tensor(ids, dtype=torch.long, device=DEVICE)
            pad_mask[i, :len(ids)] = False

        if training:
            def _fwd(ids, mask):
                return decoder(input_ids=ids, padding_mask=mask)
            hidden = torch.utils.checkpoint.checkpoint(
                _fwd, input_ids, pad_mask, use_reentrant=False
            )
        else:
            with torch.no_grad():
                hidden = decoder(input_ids=input_ids, padding_mask=pad_mask)

        mask_f = (~pad_mask).unsqueeze(-1).float()
        pooled = (hidden.float() * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)
        all_emb.append(pooled)

    return torch.cat(all_emb, dim=0)

n_lora = sum(p.numel() for p in decoder.parameters() if p.requires_grad)
n_bilstm = sum(p.numel() for p in bilstm.parameters() if p.requires_grad)
print(f"LoRA params (q,v): {n_lora/1e6:.2f}M")
print(f"BiLSTM params:     {n_bilstm/1e6:.2f}M")
print(f"Total trainable:   {(n_lora+n_bilstm)/1e6:.2f}M")

LoRA params (q,v): 0.78M
BiLSTM params:     3.81M
Total trainable:   4.59M


In [ ]:
# CELL 11 — OPTIMIZER (two LR groups) + SCHEDULER

lora_params = [p for p in decoder.parameters() if p.requires_grad]
bilstm_params = list(bilstm.parameters())

optimizer = torch.optim.AdamW([
    {"params": lora_params,   "lr": CFG.lora_lr,   "weight_decay": CFG.weight_decay},
    {"params": bilstm_params, "lr": CFG.bilstm_lr, "weight_decay": CFG.weight_decay},
])

total_steps = (len(train_loader) * CFG.epochs) // CFG.grad_accum
warmup_steps = max(1, int(total_steps * CFG.warmup_ratio))

def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return max(0.01, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler("cuda")

print(f"Total optimizer steps: {total_steps}, warmup: {warmup_steps}")
print(f"LoRA LR: {CFG.lora_lr}, BiLSTM LR: {CFG.bilstm_lr}")

Total optimizer steps: 400, warmup: 20
LoRA LR: 0.0002, BiLSTM LR: 0.0005


In [ ]:
# CELL 12 — TRAIN / EVAL / PREDICT

from sklearn.metrics import f1_score, classification_report

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=CFG.label_smoothing,
)

def train_one_epoch():
    decoder.train()
    bilstm.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0
    num_batches = len(train_loader)
    start_time = time.time()

    # Trainable parameters: LoRA adapters + BiLSTM
    trainable_params = [
        p for p in list(decoder.parameters()) + list(bilstm.parameters())
        if p.requires_grad
    ]

    optimizer.zero_grad(set_to_none=True)

    print(f"\nTraining started: {num_batches} document batches")
    print(f"Gradient accumulation: {CFG.grad_accum}")
    print("-" * 110)

    for batch_idx, doc in enumerate(train_loader, start=1):
        batch_start = time.time()

        # Each batch is one complete document
        sent_tokens = doc["sent_tokens"]
        labels = torch.tensor(
            doc["labels"], dtype=torch.long, device=DEVICE
        )

        # Encode sentences using the decoder.
        # Gradients flow through the decoder's LoRA adapters.
        with autocast(device_type="cuda", dtype=torch.float16):
            sent_emb = encode_doc_sentences(
                decoder,
                sent_tokens,
                sub_batch=CFG.sent_sub_batch,
                training=True,
            )

            # BiLSTM uses the sequence of sentence embeddings
            logits = bilstm(sent_emb.unsqueeze(0)).squeeze(0)

            # One prediction per sentence; no padded sentences here
            loss = criterion(logits, labels)

        # Gradient accumulation across documents
        scaler.scale(loss / CFG.grad_accum).backward()

        should_update = (
            batch_idx % CFG.grad_accum == 0
            or batch_idx == num_batches
        )

        if should_update:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                max_norm=CFG.max_grad_norm,
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        # Batch metrics
        with torch.no_grad():
            preds = logits.argmax(dim=-1)
            correct = (preds == labels).sum().item()
            count = labels.numel()

            total_correct += correct
            total_count += count
            total_loss += loss.item()

        # Live progress report
        batch_time = time.time() - batch_start
        elapsed = time.time() - start_time
        avg_loss = total_loss / batch_idx
        avg_acc = total_correct / max(total_count, 1)

        progress = 100.0 * batch_idx / max(num_batches, 1)
        eta = (elapsed / batch_idx) * (num_batches - batch_idx)

        status = "OPTIMIZER UPDATE" if should_update else "Accumulating gradients"

        message = (
            f"Batch {batch_idx:4d}/{num_batches} ({progress:6.2f}%) | "
            f"Doc: {doc['doc_id']} | "
            f"Sentences: {len(sent_tokens):3d} | "
            f"Loss: {loss.item():.4f} | "
            f"Avg loss: {avg_loss:.4f} | "
            f"Accuracy: {avg_acc:.2%} | "
            f"LR LoRA: {optimizer.param_groups[0]['lr']:.2e} | "
            f"LR BiLSTM: {optimizer.param_groups[1]['lr']:.2e} | "
            f"{status} | "
            f"Batch: {batch_time:.1f}s | "
            f"Elapsed: {elapsed/60:.1f}m | "
            f"ETA: {eta/60:.1f}m"
        )

        if torch.cuda.is_available():
            memory = torch.cuda.memory_allocated() / (1024 ** 3)
            message += f" | GPU RAM: {memory:.2f} GiB"

        print(message, flush=True)

    epoch_loss = total_loss / max(num_batches, 1)
    epoch_acc = total_correct / max(total_count, 1)

    print("-" * 110)
    print(
        f"TRAINING COMPLETE | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_acc:.2%} | "
        f"Time: {(time.time() - start_time)/60:.2f} min"
    )

    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate_loader(loader):
    decoder.eval(); bilstm.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    for doc in loader:
        labels = torch.tensor(doc["labels"], dtype=torch.long, device=DEVICE)
        with autocast(device_type="cuda", dtype=torch.float16):
            sent_emb = encode_doc_sentences(decoder, doc["sent_tokens"], CFG.sent_sub_batch, training=False)
            logits = bilstm(sent_emb.unsqueeze(0)).squeeze(0)
            loss = criterion(logits, labels)
        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    all_ids = list(range(NUM_LABELS))
    f1 = f1_score(all_labels, all_preds, labels=all_ids, average="macro", zero_division=0)
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / max(len(all_preds), 1)
    return total_loss / max(len(loader), 1), f1, acc, all_preds, all_labels


@torch.no_grad()
def predict_loader(loader):
    decoder.eval(); bilstm.eval()
    out = {}
    for doc in loader:
        with autocast(device_type="cuda", dtype=torch.float16):
            sent_emb = encode_doc_sentences(decoder, doc["sent_tokens"], CFG.sent_sub_batch, training=False)
            logits = bilstm(sent_emb.unsqueeze(0)).squeeze(0)
        preds = logits.argmax(dim=-1).cpu().tolist()
        out[doc["doc_id"]] = [ID2LABEL[int(p)] for p in preds]
    return out

print("Train/eval/predict ready ✓")

Train/eval/predict ready ✓


In [ ]:
# CELL 13 — GPU PREFLIGHT

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

decoder.train(); bilstm.train()
optimizer.zero_grad(set_to_none=True)

try:
    doc = next(iter(train_loader))
    labels = torch.tensor(doc["labels"], dtype=torch.long, device=DEVICE)
    print(f"Preflight doc: {doc['n_sentences']} sentences")

    with autocast(device_type="cuda", dtype=torch.float16):
        sent_emb = encode_doc_sentences(decoder, doc["sent_tokens"], CFG.sent_sub_batch, training=True)
        logits = bilstm(sent_emb.unsqueeze(0)).squeeze(0)
        loss = criterion(logits, labels)

    scaler.scale(loss).backward()
    torch.cuda.synchronize()

    peak_alloc = torch.cuda.max_memory_allocated() / 1024**3
    peak_res = torch.cuda.max_memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3

    print(f"\nFWD+BWD OK | Loss: {loss.item():.4f}")
    print(f"Peak allocated: {peak_alloc:.2f} GiB | reserved: {peak_res:.2f} GiB | total: {total:.2f} GiB")
    print(f"Headroom: {total - peak_res:.2f} GiB")
except torch.cuda.OutOfMemoryError:
    print("OOM. Reduce CFG.sent_sub_batch to 4 or CFG.sent_max_tokens to 128.")
    raise
finally:
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()

Preflight doc: 53 sentences

FWD+BWD OK | Loss: 2.5691
Peak allocated: 3.24 GiB | reserved: 3.39 GiB | total: 22.03 GiB
Headroom: 18.64 GiB


In [ ]:
# CELL 14 — VERBOSE TRAINING WITH EARLY STOPPING

best_f1 = 0.0
best_epoch = -1
no_improve = 0
history = []
BEST_CKPT = OUTPUT_DIR / "best_lora_bilstm.pt"

training_start = time.time()

print("=" * 88)
print("JOINT LoRA(q,v) + BiLSTM TRAINING")
print("=" * 88)
print(f"Planned epochs           : {CFG.epochs}")
print(f"Early-stopping patience  : {CFG.early_stop_patience}")
print(f"Checkpoint destination   : {BEST_CKPT}")
print(f"Training batches/epoch   : {len(train_loader)}")
print(f"Validation batches       : {len(dev_loader)}")
print(f"CUDA available           : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU                      : {torch.cuda.get_device_name(0)}")
    torch.cuda.reset_peak_memory_stats()

print("=" * 88)

for epoch in range(CFG.epochs):
    epoch_start = time.time()
    epoch_num = epoch + 1

    print(f"\n{'#' * 28} EPOCH {epoch_num}/{CFG.epochs} {'#' * 28}")
    print("[TRAIN] Starting training pass...")
    print("        Updating trainable LoRA(q,v) adapters and BiLSTM parameters.")

    # -------------------- TRAINING --------------------
    tr_loss, tr_acc = train_one_epoch()

    print("\n[TRAIN] Completed.")
    print(f"        Training loss     : {tr_loss:.6f}")
    print(f"        Training accuracy : {tr_acc:.4%}")

    # -------------------- VALIDATION --------------------
    print("\n[VALIDATION] Evaluating development set...")
    print("             No optimizer updates are performed during validation.")

    val_loss, val_f1, val_acc, _, _ = evaluate_loader(dev_loader)

    print("\n[VALIDATION] Completed.")
    print(f"             Validation loss     : {val_loss:.6f}")
    print(f"             Validation Macro-F1 : {val_f1:.6f}")
    print(f"             Validation accuracy: {val_acc:.4%}")

    # -------------------- METRICS AND TIMING --------------------
    lr_now = optimizer.param_groups[0]["lr"]
    epoch_seconds = time.time() - epoch_start
    elapsed_seconds = time.time() - training_start

    history.append({
        "epoch": epoch_num,
        "train_loss": tr_loss,
        "train_acc": tr_acc,
        "val_loss": val_loss,
        "val_f1": val_f1,
        "val_acc": val_acc,
        "lr": lr_now,
        "epoch_seconds": epoch_seconds,
    })

    print("\n" + "-" * 78)
    print(f"EPOCH {epoch_num} SUMMARY")
    print("-" * 78)
    print(f"  Training loss          : {tr_loss:.6f}")
    print(f"  Training accuracy      : {tr_acc:.4%}")
    print(f"  Validation loss        : {val_loss:.6f}")
    print(f"  Validation Macro-F1    : {val_f1:.6f}")
    print(f"  Validation accuracy    : {val_acc:.4%}")
    print(f"  Learning rate          : {lr_now:.3e}")
    print(f"  Epoch duration         : {epoch_seconds / 60:.2f} min")
    print(f"  Total elapsed          : {elapsed_seconds / 60:.2f} min")
    print(f"  Previous best Macro-F1 : {best_f1:.6f}")
    print(f"  Best epoch so far      : {best_epoch}")

    if torch.cuda.is_available():
        peak_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
        print(f"  Peak GPU memory        : {peak_gb:.2f} GiB")

    # -------------------- SAVE BEST CHECKPOINT --------------------
    if val_f1 > best_f1:
        previous_best = best_f1
        best_f1 = val_f1
        best_epoch = epoch_num
        no_improve = 0

        print("\n[CHECKPOINT] NEW BEST VALIDATION MACRO-F1!")
        print(f"             Previous best : {previous_best:.6f}")
        print(f"             New best      : {best_f1:.6f}")
        print(f"             Improvement   : +{best_f1 - previous_best:.6f}")
        print(f"             Saving to     : {BEST_CKPT}")

        torch.save({
            "epoch": epoch_num,
            "decoder": decoder.state_dict(),
            "bilstm": bilstm.state_dict(),
            "val_f1": val_f1,
            "val_loss": val_loss,
        }, BEST_CKPT)

        print(f"[CHECKPOINT] Saved successfully: {BEST_CKPT.exists()}")

    else:
        no_improve += 1
        remaining = CFG.early_stop_patience - no_improve

        print("\n[CHECKPOINT] No improvement this epoch.")
        print(f"             Best Macro-F1: {best_f1:.6f} (epoch {best_epoch})")
        print(f"             No-improvement streak: "
              f"{no_improve}/{CFG.early_stop_patience}")

        if remaining > 0:
            print(f"             Early stopping after {remaining} more "
                  "non-improving epoch(s), unless F1 improves.")

    # -------------------- EARLY STOPPING --------------------
    if no_improve >= CFG.early_stop_patience:
        print("\n" + "!" * 78)
        print("EARLY STOPPING TRIGGERED")
        print(f"No validation Macro-F1 improvement for "
              f"{CFG.early_stop_patience} consecutive epochs.")
        print(f"Stopping after epoch : {epoch_num}")
        print(f"Best epoch           : {best_epoch}")
        print(f"Best validation F1   : {best_f1:.6f}")
        print("!" * 78)
        break

# -------------------- FINAL SUMMARY --------------------
total_seconds = time.time() - training_start

print("\n" + "=" * 88)
print("TRAINING FINISHED")
print("=" * 88)
print(f"Epochs completed         : {len(history)}")
print(f"Best epoch               : {best_epoch}")
print(f"Best validation Macro-F1 : {best_f1:.6f}")
print(f"Total training time      : {total_seconds / 60:.2f} min")
print(f"Checkpoint path          : {BEST_CKPT}")
print(f"Checkpoint exists        : {BEST_CKPT.exists()}")

if history:
    last = history[-1]
    print("\nFinal completed epoch metrics:")
    print(f"  Training loss      : {last['train_loss']:.6f}")
    print(f"  Training accuracy  : {last['train_acc']:.4%}")
    print(f"  Validation loss    : {last['val_loss']:.6f}")
    print(f"  Validation Macro-F1: {last['val_f1']:.6f}")
    print(f"  Validation accuracy: {last['val_acc']:.4%}")

print("=" * 88)

JOINT LoRA(q,v) + BiLSTM TRAINING
Planned epochs           : 10
Early-stopping patience  : 4
Checkpoint destination   : /content/drive/MyDrive/CS781_Phase3_LoRA_BiLSTM_v2/best_lora_bilstm.pt
Training batches/epoch   : 160
Validation batches       : 10
CUDA available           : True
GPU                      : NVIDIA L4

############################ EPOCH 1/10 ############################
[TRAIN] Starting training pass...
        Updating trainable LoRA(q,v) adapters and BiLSTM parameters.

Training started: 160 document batches
Gradient accumulation: 4
--------------------------------------------------------------------------------------------------------------
Batch    1/160 (  0.62%) | Doc: CCI_Singhania_and_Partners_LLP_vs_Microsoft_CorporatioCO110039COM401821 | Sentences: 131 | Loss: 2.5754 | Avg loss: 2.5754 | Accuracy: 8.40% | LR LoRA: 0.00e+00 | LR BiLSTM: 0.00e+00 | Accumulating gradients | Batch: 3.5s | Elapsed: 0.1m | ETA: 9.2m | GPU RAM: 2.30 GiB
Batch    2/160 (  1.25%) | D

In [ ]:
# CELL 15 — DETAILED DEV EVAL

best_state = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
decoder.load_state_dict(best_state["decoder"])
bilstm.load_state_dict(best_state["bilstm"])
print(f"Loaded best epoch {best_state['epoch']}, F1={best_state['val_f1']:.4f}\n")

target_names = [ID2LABEL[i] for i in range(NUM_LABELS)]
label_ids = list(range(NUM_LABELS))

val_loss, val_f1, val_acc, val_preds, val_labels = evaluate_loader(dev_loader)
print(f"COMBINED DEV: loss={val_loss:.4f}, Macro-F1={val_f1:.4f}, acc={val_acc:.4f}\n")
print(classification_report(val_labels, val_preds, labels=label_ids,
                             target_names=target_names, digits=4, zero_division=0))

support = np.bincount(val_labels, minlength=NUM_LABELS)
scorable = [i for i in range(NUM_LABELS) if support[i] > 0]
f1_eff = f1_score(val_labels, val_preds, labels=scorable, average="macro", zero_division=0)
print(f"Macro-F1 over supported labels only ({len(scorable)}/{NUM_LABELS}): {f1_eff:.4f}")

_, cl_f1, cl_acc, _, _ = evaluate_loader(cl_dev_loader)
_, it_f1, it_acc, _, _ = evaluate_loader(it_dev_loader)
print(f"\nCL DEV: F1={cl_f1:.4f}, acc={cl_acc:.4f}")
print(f"IT DEV: F1={it_f1:.4f}, acc={it_acc:.4f}")

Loaded best epoch 7, F1=0.4449

COMBINED DEV: loss=1.4078, Macro-F1=0.4449, acc=0.5705

                        precision    recall  f1-score   support

                  Fact     0.6489    0.7117    0.6788       444
                 Issue     0.5833    0.4667    0.5185        15
    ArgumentPetitioner     0.6074    0.3675    0.4579       400
    ArgumentRespondent     0.5117    0.4816    0.4962       272
               Statute     0.6619    0.7667    0.7104       120
               Dissent     0.0000    0.0000    0.0000         0
   PrecedentReliedUpon     0.5907    0.6535    0.6206       254
PrecedentNotReliedUpon     0.2553    0.4800    0.3333        50
    PrecedentOverruled     0.0000    0.0000    0.0000         0
    RulingByLowerCourt     0.3882    0.5784    0.4646       102
    RatioOfTheDecision     0.5889    0.6079    0.5983       403
  RulingByPresentCourt     0.6047    0.5306    0.5652        49
                  None     0.4500    0.2727    0.3396        33

              

In [ ]:
# CELL 16 — TEST PREDICTIONS + SUBMISSION

predictions = predict_loader(test_loader)

assert len(predictions) == len(test_docs)
for doc_id, sentences in test_docs:
    assert doc_id in predictions
    assert len(predictions[doc_id]) == len(sentences)
    assert all(l in LABEL2ID for l in predictions[doc_id])

pred_path = OUTPUT_DIR / "prediction.json"
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

zip_path = OUTPUT_DIR / "submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, "prediction.json")

print(f"Saved: {pred_path}")
print(f"Zipped: {zip_path} ({zip_path.stat().st_size/1024:.1f} KB)")

pred_counts = Counter(l for labels in predictions.values() for l in labels)
print("\nPrediction distribution:")
for label, count in pred_counts.most_common():
    print(f"  {label:<28} {count}")

Saved: /content/drive/MyDrive/CS781_Phase3_LoRA_BiLSTM_v2/prediction.json
Zipped: /content/drive/MyDrive/CS781_Phase3_LoRA_BiLSTM_v2/submission.zip (1.5 KB)

Prediction distribution:
  RatioOfTheDecision           606
  Fact                         417
  PrecedentReliedUpon          353
  ArgumentPetitioner           193
  Statute                      182
  PrecedentNotReliedUpon       131
  ArgumentRespondent           126
  RulingByLowerCourt           103
  RulingByPresentCourt         41
  None                         27
  Issue                        17
  Dissent                      1


In [ ]:
# CELL 17 — TRAINING HISTORY

print(f"{'Epoch':>5} {'TrLoss':>8} {'TrAcc':>7} {'VaLoss':>8} {'VaF1':>8} {'VaAcc':>7}")
print("─" * 55)
for h in history:
    marker = " ★" if h["val_f1"] == best_f1 else ""
    print(f"  {h['epoch']:3d}  {h['train_loss']:>7.4f} {h['train_acc']:>6.4f} "
          f"{h['val_loss']:>7.4f} {h['val_f1']:>7.4f} {h['val_acc']:>6.4f}{marker}")
print(f"\nBest: epoch {best_epoch}, Macro-F1 {best_f1:.4f}")

Epoch   TrLoss   TrAcc   VaLoss     VaF1   VaAcc
───────────────────────────────────────────────────────
    1   2.2248 0.3137  1.8761  0.2090 0.4085
    2   1.4207 0.4750  1.7652  0.2555 0.4057
    3   1.0921 0.5840  1.4960  0.3597 0.4879
    4   0.7807 0.6499  1.4650  0.4026 0.5121
    5   0.6042 0.6995  1.4440  0.4150 0.5336
    6   0.4485 0.7697  1.4336  0.4325 0.5593
    7   0.3905 0.8111  1.4078  0.4449 0.5705 ★
    8   0.3390 0.8169  1.5046  0.4291 0.5551
    9   0.2586 0.8764  1.5180  0.4295 0.5574
   10   0.2758 0.8557  1.5117  0.4322 0.5607

Best: epoch 7, Macro-F1 0.4449
